In [ ]:
import numpy as np

In [ ]:
X = np.array(sorted([20 * x for x in np.random.rand(120, 1)]))

Y = np.array([2 * el + np.random.choice([-1, 1]) * 6 * np.random.rand() for el in X])


In [ ]:
X[-10:]

In [ ]:
Y[-10:]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.4)

In [ ]:
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

In [ ]:
### Сохраняем упорядоченные индексы наших элементов
index_argsort = np.argsort(X_train.reshape(72, ))


In [ ]:
index_argsort

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
### Строим простую модель
model = LinearRegression(fit_intercept=False)
model.fit(X_train, Y_train)

In [ ]:
model.predict(X_train)

In [ ]:
model.predict(X_test)

In [ ]:
### Изображаем
import matplotlib.pyplot as plt

In [ ]:
### Установим красивые дефолтные настройки
### Может быть лень постоянно прописывать
### У графиков параметры цвета, размера, шрифта
### Можно положить их в словарь дефолтных настроек

import matplotlib as mlp

mlp.rcParams['lines.linewidth'] = 5
mlp.rcParams['xtick.major.size'] = 20
mlp.rcParams['xtick.major.width'] = 5
mlp.rcParams['xtick.labelsize'] = 20
mlp.rcParams['xtick.color'] = '#FF5533'

mlp.rcParams['ytick.major.size'] = 20
mlp.rcParams['ytick.major.width'] = 5
mlp.rcParams['ytick.labelsize'] = 20
mlp.rcParams['ytick.color'] = '#FF5533'

mlp.rcParams['axes.labelsize'] = 20
mlp.rcParams['axes.titlesize'] = 20
mlp.rcParams['axes.titlecolor'] = '#00B050'
mlp.rcParams['axes.labelcolor'] = '#00B050'

In [ ]:
fig = plt.figure()
fig.set_size_inches(14, 12)

plt.scatter(X_train, Y_train, c='#00B050', s=100)
plt.scatter(X_test, Y_test, c='#FF5533', s=100)
plt.plot(X, [2 * x for x in X], '#1E2027', linewidth=4)
plt.plot(X_train[index_argsort], model.predict(X_train[index_argsort]), '--g', linewidth=4)

plt.legend(['Тренировочная выборка', 'Тестовая выборка', 'True function',
            'Линейная модель'], loc='upper left')
plt.xlabel('X')
plt.ylabel('Y')

plt.show()

In [ ]:
model.coef_

In [ ]:
### Построим полиномиальную модель

X_pol = X_train.copy()

for k in range(2, 26):
    X_pol = np.append(X_pol, np.array([x ** k for x in X_pol[:, 0]]).reshape(72, -1),
                      axis=1)

In [ ]:
X_pol[0]

In [ ]:
model_pol = LinearRegression()
model_pol.fit(X_pol, Y_train)

In [ ]:
### Изобразим!
import matplotlib.pyplot as plt

fig = plt.figure()
fig.set_size_inches(14, 12)

plt.scatter(X_train, Y_train, c='#00B050', s=100)
plt.scatter(X_test, Y_test, c='#FF5533', s=100)
plt.plot(X, [2 * x for x in X], '#1E2027', linewidth=4)
plt.plot(X_train[index_argsort],
         model.predict(X_train[index_argsort]),
         '--g', linewidth=2)
plt.plot(X_train[index_argsort],
         model_pol.predict(X_pol[index_argsort]),
         '--r', linewidth=2)

plt.legend(['Тренировочная выборка', 'Тестовая выборка', 'True function',
            'Линейная модель', 'Оценка полиномиальной модели'],
           loc='upper left')
plt.xlabel('X')
plt.ylabel('Y')

plt.title('Графическая иллюстрация переобучения и как с ним можно бороться')

plt.show()

In [ ]:
np.mean((model.predict(X_train) - Y_train) ** 2)

In [ ]:
np.mean((model.predict(X_test) - Y_test) ** 2)

In [ ]:
np.mean((model_pol.predict(X_pol) - Y_train) ** 2)

In [ ]:
X_pol_test = X_test.copy()

for k in range(2, 26):
    X_pol_test = np.append(X_pol_test,
                           np.array([x ** k for x in X_pol_test[:, 0]]).reshape(48, -1),
                           axis=1)

In [ ]:
np.mean((model_pol.predict(X_pol_test) - Y_test) ** 2)

## Кросс валидация

## Регуляризация полиномиальной модели

In [ ]:
X_pol

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaler.fit(X_pol)

In [ ]:
X_pol_transformed = scaler.transform(X_pol)
X_pol_transformed_test = scaler.transform(X_pol_test)

In [ ]:
from sklearn.linear_model import Ridge

model_ridge = Ridge()
model_ridge.fit(X_pol_transformed, Y_train)

predictions_ridge_train = model_ridge.predict(X_pol_transformed)
predictions_ridge_test = model_ridge.predict(X_pol_transformed_test)

error_train = np.mean((predictions_ridge_train - Y_train) ** 2)
error_test = np.mean((predictions_ridge_test - Y_test) ** 2)

print(f"Качество Ridge полиномиальной регрессии на трейне: {round(error_train, 2)}")
print(f"Качество Ridge полиномиальной регрессии на тесте: {round(error_test, 2)}")